# PPT Generator Agent

The PPT Generator Agent converts any script, lecture notes, or written content into a complete, self-contained HTML presentation in the Jobs-style minimalist aesthetic — dark backgrounds, white typography, and the "one screen, one idea" principle.


In [1]:
!pip install -q aixplain

## Add your aiXplain access key

Get your aiXplain access key from the [Integrations](https://platform.aixplain.com/account/integrations) page.

In [ ]:

AixplainAPIKey = "<YOUR_AIXPLAIN_API_KEY>" 

import os
os.environ["TEAM_API_KEY"] = AixplainAPIKey


In [ ]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)


# What is the PPT Generator Agent?

The goal of this agent is to turn raw written content — scripts, notes, articles, or bullet points — into a polished, ready-to-open HTML slide deck with no design tools or extra software required.

## Agent Architecture

The agent uses one custom script tool:

- **Presentation Planner** — parses the input script and returns a structured JSON outline with slide types, titles (≤12 characters, Jobs-style), key points, and speaker notes for each slide.

The agent then uses this outline to generate the complete self-contained HTML file.

## Agent Workflow

1. **Parse** — call the Presentation Planner tool to break the script into a slide-by-slide structure.
2. **Refine** — apply Jobs-style title constraints and select appropriate slide types.
3. **Generate** — produce a single self-contained HTML file with Tailwind CSS, animations, and dark styling.
4. **Deliver** — return the complete HTML ready to open in any browser.


In [5]:
def plan_presentation_structure(script: str, page_count: int = 12) -> str:
    """
    Parses a script or notes and returns a structured slide-by-slide outline
    for a Jobs-style minimalist presentation.

    Parameters:
        script (str): The raw script, lecture notes, or content to present.
        page_count (int): Target number of slides (default: 12, range: 8-20).

    Returns:
        str: A JSON-formatted slide outline with slide number, type, title,
             key point, and speaker note for each slide.
    """
    import json
    import re

    page_count = max(8, min(page_count, 20))

    # Split script into meaningful segments
    lines = [l.strip() for l in script.strip().splitlines() if l.strip()]
    paragraphs = []
    current = []
    for line in lines:
        if line.endswith(('.', '?', '!')) or len(line) > 80:
            current.append(line)
            paragraphs.append(' '.join(current))
            current = []
        else:
            current.append(line)
    if current:
        paragraphs.append(' '.join(current))

    if not paragraphs:
        paragraphs = [script[:200]]

    # Distribute paragraphs across slides
    step = max(1, len(paragraphs) // max(page_count - 2, 1))
    content_chunks = [paragraphs[i:i+step] for i in range(0, len(paragraphs), step)]
    content_chunks = content_chunks[:page_count - 2]  # reserve cover + closing

    SLIDE_TYPES = ["cover", "statement", "quote", "list", "comparison",
                   "stat", "image-text", "closing"]

    def shorten_title(text, max_chars=12):
        words = text.split()
        title = ""
        for w in words:
            if len(title) + len(w) + 1 <= max_chars:
                title = (title + " " + w).strip()
            else:
                break
        return title.upper() if title else text[:max_chars].upper()

    slides = []

    # Cover slide
    first_line = paragraphs[0] if paragraphs else "Presentation"
    slides.append({
        "slide": 1,
        "type": "cover",
        "title": shorten_title(first_line),
        "key_point": first_line[:100],
        "speaker_note": "Opening — set the stage.",
        "background": "#000000",
    })

    # Content slides
    for idx, chunk in enumerate(content_chunks, start=2):
        text = ' '.join(chunk)
        slide_type = SLIDE_TYPES[idx % len(SLIDE_TYPES)]
        if slide_type == "cover":
            slide_type = "statement"
        slides.append({
            "slide": idx,
            "type": slide_type,
            "title": shorten_title(text),
            "key_point": text[:150],
            "speaker_note": text[:200],
            "background": "#000000" if idx % 2 == 0 else "#0a0a0a",
        })

    # Closing slide
    slides.append({
        "slide": len(slides) + 1,
        "type": "closing",
        "title": "THANK YOU",
        "key_point": "Questions & Discussion",
        "speaker_note": "Close with a call to action or Q&A.",
        "background": "#000000",
    })

    return json.dumps({"total_slides": len(slides), "slides": slides}, indent=2, ensure_ascii=False)


In [6]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

planner_tool = aix.Tool(
    name="Presentation Planner",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": inspect.getsource(plan_presentation_structure), "function_name": "plan_presentation_structure"},
)
planner_tool.save()
print(f"Created: {planner_tool.name}")


Created: Presentation Planner


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, a custom script tool that parses content into a structured slide outline.
* The **instructions** that guide the agent's behaviour and HTML generation.


In [7]:
ppt_agent = aix.Agent(
    name="PPT Generator Agent",
    description="Converts scripts and notes into self-contained Jobs-style HTML presentations.",
    instructions="""
    You are an expert presentation designer specialising in Jobs-style minimalist slide decks.
    You convert any script or written content into a complete, self-contained HTML presentation.

    ## Workflow
    1. Call the Presentation Planner tool with the user's script to get a structured slide outline.
    2. Use the outline to generate a complete HTML file following the design rules below.
    3. Return ONLY the full HTML code — no explanation, no markdown fences, just raw HTML.

    ## Design Rules
    - One screen, one idea. Never crowd a slide.
    - Background: #000000 or #0a0a0a only.
    - Text: #ffffff primary, #999999 secondary.
    - Titles: ALL CAPS, maximum 12 characters, large font (clamp 3rem to 8vw).
    - Body text: maximum 2 short lines per slide.
    - No bullet lists on content slides — use statement or quote format instead.
    - Fonts: Inter or Roboto via CDN.
    - Aspect ratio: 9:16 vertical (mobile-first).
    - Navigation: keyboard arrow keys + on-screen prev/next buttons.
    - Slide transitions: CSS fade or slide-up (150-250ms ease).
    - First slide: cover with title + subtitle only.
    - Last slide: closing with "THANK YOU" or a strong call to action.

    ## HTML Template Structure
    - Single .html file, fully self-contained.
    - Tailwind CSS via CDN: https://cdn.tailwindcss.com
    - Each slide is a <section> with full viewport height (100vh).
    - JavaScript handles slide visibility and keyboard navigation.
    - Include a slide counter (e.g. "3 / 12") in the bottom-right corner.

    ## Slide Types
    - cover: large centred title + one-line subtitle
    - statement: single bold sentence, centred, large
    - quote: italic quote text + attribution
    - stat: giant number/metric + short label
    - list: max 3 items, left-aligned, large spacing
    - image-text: left text + right placeholder for image
    - closing: "THANK YOU" + CTA or contact line
    """,
    tools=[planner_tool],
)

print(f"Agent created")


Agent created: 69c43e61228f69075764f166


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.


In [8]:
script = """
Artificial intelligence is transforming every industry.
From healthcare to finance, AI is automating complex tasks and uncovering insights humans miss.
The key shift is from narrow AI tools to autonomous agents that can reason, plan, and act.
Large language models are the foundation — but the real power comes from giving them tools and memory.
In 2025, the biggest opportunity is building AI agents that work end-to-end without human intervention.
Companies that invest in agentic AI today will have a 10x productivity advantage within three years.
The barrier to entry has never been lower. You do not need a PhD. You need curiosity and the right platform.
Start small. Pick one repetitive workflow. Automate it with an agent. Measure the result. Then scale.
"""

result = ppt_agent.run(f"Generate a presentation from this script:\n{script}")


In [9]:
print(result.data.output)


<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@700&display=swap" rel="stylesheet">
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <title>AI Presentation</title>
    <style>
        body { background-color: #0a0a0a; color: #ffffff; font-family: 'Inter', sans-serif; }
        section { height: 100vh; display: flex; justify-content: center; align-items: center; flex-direction: column; text-align: center; }
        .title { font-size: clamp(3rem, 8vw, 8rem); text-transform: uppercase; }
        .key-point { font-size: 2rem; max-width: 600px; }
        .counter { position: absolute; bottom: 20px; right: 20px; color: #999999; }
        .hidden { display: none; }
    </style>
</head>
<body>
    <section id="slide1">
        <div class="title">ARTIFICIAL</div>
        <div

In [10]:
# Save the HTML output to a file and open it in your browser
html_output = result.data.output

# Strip any markdown fences if present
if "```html" in html_output:
    html_output = html_output.split("```html")[1].split("```")[0].strip()
elif "```" in html_output:
    html_output = html_output.split("```")[1].split("```")[0].strip()

with open("presentation.html", "w") as f:
    f.write(html_output)

print("Saved to presentation.html — open it in any browser.")


Saved to presentation.html — open it in any browser.


In [11]:
result = ppt_agent.run(
    "Generate a 6-slide pitch deck from this:\n"
    "We are building an AI-powered hiring platform that reduces time-to-hire by 80%.\n"
    "Our product screens candidates, schedules interviews, and drafts offer letters automatically.\n"
    "We have 12 paying customers, $40k MRR, and are raising a $2M seed round."
)
print(result.data.output)


<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@700&display=swap" rel="stylesheet">
    <link href="https://cdn.jsdelivr.net/npm/tailwindcss@2.2.19/dist/tailwind.min.css" rel="stylesheet">
    <title>AI-Powered Hiring Platform Pitch Deck</title>
    <style>
        body { font-family: 'Inter', sans-serif; background-color: #0a0a0a; color: #ffffff; }
        section { height: 100vh; display: flex; justify-content: center; align-items: center; flex-direction: column; text-align: center; }
        .title { font-size: clamp(3rem, 8vw, 8rem); text-transform: uppercase; }
        .key-point { font-size: 2rem; }
        .counter { position: absolute; bottom: 20px; right: 20px; color: #999999; }
        .hidden { display: none; }
    </style>
</head>
<body>
    <section>
        <div class="title">WE ARE</div>
        <div class="key-

# Save the Agent

Once you are happy with your agent, save it to access the agent endpoints.


In [12]:
ppt_agent.save()


Agent(path=asma-furniturewala/ppt-generator-agent/aixplain)

aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).
